In [ ]:
#importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
#loading the data set and having abrief overvuiew of how it looks
df = pd.read_csv("expiry_price_data.csv")
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.dtypes.value_counts())
print(df.describe())

In [ ]:
#Checking for null values, duplicates and outliers
print('Checking for null values....')
print(df.isnull().sum())
print('Checking for duplicates....')
print(df.duplicated().sum())

In [ ]:
#displaying the first 20 inputs of the dataset

print(df.head(20))

In [ ]:
#Checking for outliers

# Select numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

print("Numerical columns:", list(numerical_cols))
print("\nOutlier detection using IQR method:")

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # Detect outliers
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    num_outliers = outliers.shape[0]
    
    print(f"{col}: {num_outliers} outlier(s)")
    
    # Plot boxplot
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col} (Outliers: {num_outliers})')
    plt.show()

In [ ]:
#disposal variable
df['Disposal'] = np.where(df['Used_Duration'] >= df['Expiry_Years'], 1, 0)
#encoding categorical data
le = LabelEncoder()

categorical_cols = ['Product_Type','Brand','Build_Quality','Usage_Pattern','Condition']

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])
    #define features and target
    X = df[['Product_Type','Brand','Build_Quality','User_Lifespan',
        'Usage_Pattern','Expiry_Years','Condition',
        'Original_Price','Used_Duration','Current_Price']]

y = df['Disposal']


In [ ]:
#test train spliting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
#train
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

In [ ]:
y_pred = knn.predict(X_test)

In [ ]:
#model evaluation
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

#Confusion matrix
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
#report
print(classification_report(y_test, y_pred))

In [ ]:
#predict new device
new_device = [[2,1,2,5,1,4,2,1200,5,150]]

new_device = scaler.transform(new_device)

prediction = knn.predict(new_device)

if prediction == 1:
    print("Device is READY for disposal")
else:
    print("Device is NOT ready for disposal")

In [ ]:
#visualize K value optimization
error_rates = []

for i in range(1,20):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train,y_train)
    pred_i = knn.predict(X_test)
    error_rates.append(np.mean(pred_i != y_test))

plt.plot(range(1,20), error_rates)
plt.title("K Value vs Error Rate")
plt.xlabel("K")
plt.ylabel("Error Rate")
plt.show()

In [ ]:
#devices ready for disposal
sns.countplot(x=df["Disposal"])
plt.title("Devices Ready for Disposal vs Not")
plt.show()

In [ ]:
#naive bayes
from sklearn.naive_bayes import GaussianNB
#training
nb_model = GaussianNB()

nb_model.fit(X_train, y_train)

In [ ]:
#predictions
nb_predictions = nb_model.predict(X_test)
#Evaluate model
from sklearn.metrics import accuracy_score

nb_accuracy = accuracy_score(y_test, nb_predictions)

print("Naive Bayes Accuracy:", nb_accuracy)

In [ ]:
#confusion matrix for naive bayes
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm_nb = confusion_matrix(y_test, nb_predictions)

sns.heatmap(cm_nb, annot=True, fmt='d')
plt.title("Naive Bayes Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
#classification report
from sklearn.metrics import classification_report

print(classification_report(y_test, nb_predictions))

In [ ]:
#comparison
print("KNN Accuracy:", accuracy)
print("Naive Bayes Accuracy:", nb_accuracy)

In [ ]:
#Predicted class distribution
pred_series = pd.Series(nb_predictions)

sns.countplot(x=pred_series)
plt.title("Naive Bayes Predicted Disposal Classes")
plt.xlabel("Prediction (0 = Not Disposal, 1 = Disposal)")
plt.show()